## What a CNI plugin actually does — second-by-second

When the kubelet creates a pod (notebook 08), the networking side unfolds in milliseconds:

1. **The kubelet creates the pod's network namespace** (via the runtime over CRI) — an empty net namespace with only a loopback.
2. **The kubelet invokes the CNI plugin binary**, passing the namespace, container ID, and the config from `/etc/cni/net.d/`.
3. **The plugin allocates an IP** from the node's slice of the pod CIDR (via an IPAM plugin — `host-local` or `calico-ipam`).
4. **The plugin creates a `veth` pair** — one end named `eth0` inside the pod, the other on the host.
5. **The plugin programs routes** — pod-side, host-side, and any overlay/BGP fabric. The pod can now reach out.
6. **The plugin returns the IP** — the kubelet records it in `status.podIP`.
7. **`kube-proxy` notices the pod goes `Ready`** (after probes) and updates Service endpoints; DNS starts returning this IP.
8. **The containers start** — they see `eth0` already configured.

Steps 1–6 take milliseconds. Whatever the CNI does — VXLAN encapsulation, BGP advertisement, cloud VPC route, eBPF attachment — it does fast.

Inspect it: `kubectl get pod -o wide` (the IP), `kubectl exec <pod> -- ip addr` (the pod's view), `ip link` on the node (the host end of each `veth`, named `vethXXXX` or `caliXXXX`). On our map this is the **CRI** (and its sibling CNI) on the worker node — the mechanism that turns "a Pod was scheduled here" into "a Pod with a real, wired-up IP."